<a href="https://colab.research.google.com/github/Sai-charan822/DSA03-NLP/blob/main/LAB/EXP8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Experiment 8: Simple Stochastic Part-of-Speech Tagging
This program demonstrates how to implement a basic stochastic (probabilistic) POS tagger using Python. It computes transition and emission probabilities from a small training corpus and uses them to tag new sentences.

In [13]:
from collections import defaultdict, Counter

# 1. Define a small training corpus (Word, Tag)
training_corpus = [
    [('The', 'DET'), ('dog', 'NOUN'), ('ran', 'VERB')],
    [('The', 'DET'), ('cat', 'NOUN'), ('slept', 'VERB')],
    [('A', 'DET'), ('dog', 'NOUN'), ('barked', 'VERB')],
    [('The', 'DET'), ('man', 'NOUN'), ('saw', 'VERB'), ('the', 'DET'), ('dog', 'NOUN')]
]

# 2. Calculate frequencies
tag_transitions = defaultdict(Counter)  # tag_i-1 -> tag_i
tag_emissions = defaultdict(Counter)    # tag -> word
tag_counts = Counter()

# Process the corpus to gather counts
for sentence in training_corpus:
    prev_tag = "START"
    tag_counts[prev_tag] += 1
    for word, tag in sentence:
        tag_emissions[tag][word.lower()] += 1
        tag_transitions[prev_tag][tag] += 1
        tag_counts[tag] += 1
        prev_tag = tag

# 3. Compute Probabilities
# Transition Probability: P(Tag_i | Tag_i-1)
transition_prob = {}
for prev_tag, next_tags in tag_transitions.items():
    total_transitions = sum(next_tags.values())
    transition_prob[prev_tag] = {tag: count / total_transitions for tag, count in next_tags.items()}

# Emission Probability: P(Word_i | Tag_i)
emission_prob = {}
for tag, words in tag_emissions.items():
    total_emissions = sum(words.values())
    emission_prob[tag] = {word: count / total_emissions for word, count in words.items()}

# 4. Stochastic POS Tagger Function
def stochastic_tagger(sentence):
    words = [w.lower() for w in sentence.split()]
    tagged_sentence = []
    prev_tag = "START"

    all_tags = list(tag_emissions.keys())

    for word in words:
        best_tag = None
        max_prob = -1

        for tag in all_tags:
            # P(Word | Tag)
            p_emission = emission_prob[tag].get(word, 1e-5) # Use a small epsilon for unseen words
            # P(Tag | Prev_Tag)
            p_transition = transition_prob.get(prev_tag, {}).get(tag, 1e-5)

            # Combined Probability (stochastic score)
            combined_prob = p_emission * p_transition

            if combined_prob > max_prob:
                max_prob = combined_prob
                best_tag = tag

        tagged_sentence.append((word, best_tag))
        prev_tag = best_tag

    return tagged_sentence

# Test the stochastic tagger
test_sentence = "The cat saw a dog"
results = stochastic_tagger(test_sentence)

print("=== Experiment 8: Stochastic POS Tagging ===")
print(f"Test Sentence: \"{test_sentence}\"")
print("Tagged Output:", results)


=== Experiment 8: Stochastic POS Tagging ===
Test Sentence: "The cat saw a dog"
Tagged Output: [('the', 'DET'), ('cat', 'NOUN'), ('saw', 'VERB'), ('a', 'DET'), ('dog', 'NOUN')]
